# MODELO DE RUTEO — copia CORREGIDA (v2)

> ⚠ **Corregido: re-ejecutar.** Copia del notebook original con las correcciones **E1–E5** aplicadas y las
> salidas limpiadas. La lógica corregida está portada al núcleo probado `app/core/` (ruteo, comparación) y
> verificada por `tests/test_correcciones.py`. **No** contiene cifras pre-ejecutadas: re-ejecutar para regenerar.

| ID | Corrección | Celda |
|----|-----------|-------|
| E1 | `% de victorias` invertido | 28 |
| E2 | ruta abierta: ventana del último cliente | 18, 20, 23, 25 |
| E3 | calibración β sin estratificación | 22 |
| E4 | ancla silueta `1.6·k_cap` → `k_cap` | 15 |

> ⚠ El notebook **no se ejecutó** al generar esta copia; los resultados mostrados provienen del núcleo `app/core/`.


# Modelado de ruteo — K-Means + Nearest Neighbor (Alt. A) vs K-Means + ACO (Alt. B)

**Distribuidora Hijos de León (Boyacá y Cundinamarca)**

Este notebook compara dos alternativas de construcción de ruteros partiendo de la misma base limpia
(`base_clientes_limpia.csv`) y del mismo agrupamiento geográfico:

| | **Alternativa A** | **Alternativa B** |
|---|---|---|
| Agrupamiento | K-Means (igual en ambas) | K-Means (mismo que A) |
| Secuenciación | **Nearest Neighbor** (voraz) | **Colonia de hormigas (ACO)** + 2-opt |
| Naturaleza | Rápido, greedy, una pasada | Búsqueda metaheurística, más elaborada |

**Flujo del notebook**

1. Universo de ruteo (solo clientes activos, con asesor y CD).
2. Feature engineering: depósito por centro de distribución, proyección a km, ventanas horarias, capacidad y cobertura quincenal.
3. División entrenamiento / prueba 80-20 (K-Means se ajusta **solo en train** y asigna a test con `predict`).
4. K-Means por grupo (Asesor × CD), con número de sub-rutas respetando la capacidad diaria.
5. Alternativa A: Nearest Neighbor.
6. Alternativa B: ACO con lista de k-vecinos + 2-opt.
7. Comparación A vs B: distancia, tiempo de cómputo y restricciones incumplidas, con experimento multi-semilla y prueba de Wilcoxon (patrón de los ejemplos del curso).

**Restricciones de negocio modeladas**
- *Capacidad*: máximo `CAP` visitas por ruta/día.
- *Ventana horaria*: tiendas/bodegas en la mañana, entretemiento en la tarde, otros flexible.
- *Cobertura quincenal*: el cliente debe quedar en las dos quincenas (Q1-Q2) para lograr 2 visitas/mes ~15 días separadas.


## 1. Entorno y carga de datos


In [ ]:
import os, re, time, warnings, unicodedata
import itertools

# --- Detección de entorno: ¿Google Colab o local? ---
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False
print("Entorno:", "Google Colab" if IN_COLAB else "Local (.venv)")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score
from scipy.stats import wilcoxon
from geopy.distance import geodesic

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100

# --- Semilla global y parámetros configurables ---
SEMILLA = 42
np.random.seed(SEMILLA)

CAP = 45                      # máximo de visitas por ruta/día (punto medio del rango 40-50 del EDA)
UMBRAL_GRUPO = 20             # mínimo de clientes para que un par Asesor×CD sea una ruta viable
TEST_SIZE = 0.20

# Parámetros ACO (se ajustan de forma menor en la sección 6)
ACO = dict(m=12, alfa=1.0, beta=3.0, rho=0.30, Q=1.0, iteraciones=40)
ACO_SEEDS = dict(m=8, iteraciones=25)   # parámetros más ligeros para el experimento multi-semilla
N_SEEDS = 10                             # semillas del experimento de estabilidad (sube para más rigor, más lento)

print(f"Semilla global: {SEMILLA} | Capacidad/ruta: {CAP} | Test: {int(TEST_SIZE*100)}%")
print("ACO:", ACO)


In [ ]:
# --- Carga de la base limpia (raíz o data/, con widget en Colab) ---
def cargar_base(nombre):
    if IN_COLAB:
        subido = files.upload()
        return pd.read_csv(list(subido.keys())[0], encoding="utf-8-sig")
    for c in [nombre, os.path.join("data", nombre)]:
        if os.path.exists(c):
            return pd.read_csv(c, encoding="utf-8-sig")
    raise FileNotFoundError(f"No encontré {nombre}. Cópialo en la raíz o en data/.")

df = cargar_base("base_clientes_limpia.csv")
print("Base cargada:", df.shape)

# --- Nombre del centro de distribución (ya viene en la base limpia exportada por el EDA) ---
if "Nombre CD" not in df.columns:
    df["Nombre CD"] = "CD " + df["Cod CD"].astype(str)   # respaldo si se usa una base limpia antigua

print("Centros de distribución en la base:", df["Cod CD"].nunique())
df[["CodigoCliente", "Supervisor", "Cod CD", "Nombre CD", "Municipio", "Latitud", "Longitud"]].head()


## 2. Universo de ruteo

Filtramos a los clientes que de verdad se rutear:

- `Supervisor` con asesor real (se excluye `SIN_ASESOR`, el antiguo `'--'` de la "Unidad Bolsa C2D").
- `activo == 1` (solo clientes activos).
- `visitas_mes_rep > 0` (con visita asignada).

Además se validan las coordenadas (que caigan dentro del área de Boyacá/Cundinamarca) y se guarda el **embudo** de filas.


In [ ]:
df0 = df.copy()
embudo = [("Base limpia completa", len(df))]

# --- Filtros del universo de ruteo ---
df = df[df["Supervisor"].astype(str).str.strip().ne("SIN_ASIGNAR")].copy()
embudo.append(("Con asesor asignado", len(df)))

df = df[df["activo"] == 1].copy()
embudo.append(("Solo activos", len(df)))

df = df[df["visitas_mes_rep"] > 0].copy()
embudo.append(("Con visita asignada (visitas_mes_rep>0)", len(df)))

# --- Validación de coordenadas (caja de Boyacá/Cundinamarca) ---
dentro = df["Latitud"].between(3.0, 6.2) & df["Longitud"].between(-77.0, -72.5)
df = df[dentro].copy()
embudo.append(("Con coordenadas válidas", len(df)))

emb = pd.DataFrame(embudo, columns=["paso", "filas"])
emb["eliminadas"] = (-emb["filas"].diff()).fillna(0).astype(int)
display(emb)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(emb["paso"][::-1], emb["filas"][::-1], color="#2e86c1")
for i, v in enumerate(emb["filas"][::-1]):
    ax.text(v + 80, i, f"{v:,}", va="center")
ax.set_title("Embudo del universo de ruteo"); ax.set_xlabel("clientes")
plt.tight_layout(); plt.show()


### 2.1 Grupos de ruteo (Asesor × Centro de Distribución)

Cada asesor opera desde uno o más centros de distribución (CD). El **CD es el depósito** del que sale y al que regresa la ruta. El grupo `(Supervisor, Cod CD)` define el territorio; se conservan los grupos con al menos `UMBRAL_GRUPO` clientes.

El **depósito** de cada grupo es el **centroide** (media de lat/long) de sus clientes, porque la base no trae las coordenadas exactas del CD. Para los CD "estadísticos" (AV58/AV59/AV61) esta aproximación es la misma que para los físicos.


In [ ]:
g = (df.groupby(["Supervisor", "Cod CD"])
       .agg(n=("CodigoCliente", "size"),
            nombre_cd=("Nombre CD", "first"),
            municipios=("Municipio", "nunique"))
       .reset_index()
       .sort_values("n", ascending=False))
g["viable"] = np.where(g["n"] >= UMBRAL_GRUPO, "sí", "no")
display(g.style.format({"n": "{:,}"}))

grupos_ok = g[g["viable"] == "sí"][["Supervisor", "Cod CD"]].values.tolist()
df["grupo_id"] = df["Supervisor"] + " | " + df["Cod CD"]
keep = {(s, c) for s, c in grupos_ok}
df = df[df[["Supervisor", "Cod CD"]].apply(tuple, axis=1).isin(keep)].copy()

print(f"Grupos viables: {len(grupos_ok)} | Clientes en grupos viables: {len(df):,}")
por_cd = df.groupby(["Cod CD", "Nombre CD"]).size().reset_index(name="clientes").sort_values("clientes", ascending=False)
display(por_cd.style.format({"clientes": "{:,}"}))


## 3. Feature engineering

Preparo las variables que usan los dos modelos:

1. **Depósito por grupo**: centroide de lat/long de sus clientes.
2. **Proyección local a kilómetros**: equirectangular con la latitud media del grupo (`x = R·cos(φ₀)·Δlon`, `y = R·Δlat`). Da distancias euclídeas en km, aptas para K-Means y para las heurísticas de NN/ACO. La distancia que se **reporta** es haversina/geoepica sobre lat/long originales.
3. **Ventana horaria** por `CategoriaActividad` (slot 0 = mañana, 1 = mediodía/flexible, 2 = tarde): tiendas y bodegas en la mañana, entretenimiento en la tarde, otros flexible.
4. **Capacidad** `CAP` por ruta y número de sub-rutas por grupo.
5. **Cobertura quincenal** (`FRE_REP == "Q1-Q2"`): cliente que logra las 2 visitas/mes ~15 días separadas.


In [ ]:
R_TIERRA = 6371.0  # km

# --- Ventanas horarias: conjunto de slots permitidos por cliente ---
VENTANAS = {"Tiendas": {0}, "Bodegas": {0}, "Entretenimiento": {2}, "Otros": {0, 1, 2}}
df["slots"] = df["CategoriaActividad"].map(lambda c: VENTANAS.get(c, {0, 1, 2}))

# --- Cobertura quincenal (2 visitas/mes ~15 días) ---
df["cubre_quincenas"] = (df["FRE_REP"] == "Q1-Q2").astype(int)

# --- Proyección a km por grupo (φ₀ = latitud media del grupo) ---
# Se usa transform (vectorizado) para no perder columnas al agrupar (pandas 3.x excluye la
# columna de agrupación en .apply, así que transform es más seguro aquí).
phi0 = np.radians(df.groupby("grupo_id")["Latitud"].transform("mean"))
df["x_km"] = R_TIERRA * np.radians(df["Longitud"]) * np.cos(phi0)
df["y_km"] = R_TIERRA * np.radians(df["Latitud"])

# --- Depósito por grupo (centroide de sus clientes) ---
dep = (df.groupby("grupo_id")
         .agg(dep_lat=("Latitud", "mean"), dep_lon=("Longitud", "mean"),
              n=("CodigoCliente", "size"), nombre_cd=("Nombre CD", "first"))
         .reset_index())
dep["n_rutas"] = np.ceil(dep["n"] / CAP).astype(int)
display(dep.style.format({"dep_lat": "{:.4f}", "dep_lon": "{:.4f}", "n": "{:,}"}))

print("Total de sub-rutas estimadas (por capacidad):", int(dep["n_rutas"].sum()))
print("\nDistribución de ventana horaria (categoría de actividad):")
display(df["CategoriaActividad"].value_counts().to_frame("clientes"))
print(f"Cobertura quincenal (Q1-Q2, cumple 2 visitas/mes): "
      f"{df['cubre_quincenas'].sum():,} de {len(df):,} ({df['cubre_quincenas'].mean()*100:.1f}%)")


### 3.1 Matrices de distancia

La distancia haversina es la que se reporta; la proyección a km se usa dentro de K-Means/NN/ACO por velocidad. Aquí verifico que ambas coinciden dentro de un margen despreciable en la escala de estos territorios.


In [ ]:
def haversine_mat(lat, lon):
    """Matriz de distancias haversina (km) para arreglos de lat/long."""
    la = np.radians(np.asarray(lat, float))[:, None]
    lo = np.radians(np.asarray(lon, float))[:, None]
    dla = la - la.T
    dlo = lo - lo.T
    a = np.sin(dla/2)**2 + np.cos(la)*np.cos(la.T)*np.sin(dlo/2)**2
    return 2 * R_TIERRA * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# Verificación en un grupo de ejemplo
ejemplo = df[df["grupo_id"] == df["grupo_id"].value_counts().index[0]].head(200)
Dh = haversine_mat(ejemplo["Latitud"], ejemplo["Longitud"])
xe, ye = ejemplo["x_km"].values, ejemplo["y_km"].values
Dp = np.sqrt((xe[:,None]-xe[None,:])**2 + (ye[:,None]-ye[None,:])**2)
iu = np.triu_indices_from(Dh, 1)
diff = np.abs(Dh - Dp)[iu]
print(f"Grupo de ejemplo (n={len(ejemplo)}):")
print(f"  Distancia haversina vs proyectada — diferencia media {diff.mean():.4f} km, "
      f"máxima {diff.max():.4f} km ({100*diff.max()/Dh[iu].max():.2f}% de la mayor distancia)")


## 4. División entrenamiento / prueba (80-20)

Como el K-Means es el paso "entrenable", se ajusta **solo con el 80 % de los clientes (train)** y el 20 % restante (test) se asigna a los centros aprendidos con `predict`. Así evitamos fugas (*leakage*) y podemos medir qué tan bien generaliza el agrupamiento a clientes no vistos. La división se hace **estratificada por asesor** para mantener el balance de territorios.

> Las dos alternativas (A y B) se evalúan sobre el **mismo** split y los mismos sub-rutas, para que la comparación sea justa.


In [ ]:
train, test = train_test_split(df, test_size=TEST_SIZE, random_state=SEMILLA, stratify=df["Supervisor"])
train = train.copy(); test = test.copy()
print(f"Train: {len(train):,} ({len(train)/len(df)*100:.0f}%) | Test: {len(test):,} ({len(test)/len(df)*100:.0f}%)")
print("\nReparto por asesor (estratificado):")
display(pd.DataFrame({"train": train["Supervisor"].value_counts(),
                      "test": test["Supervisor"].value_counts(),
                      "% test": (test["Supervisor"].value_counts()/train["Supervisor"].value_counts()*100).round(1)}))


## 5. K-Means por grupo (común a A y B)

Para cada grupo (Asesor × CD) se ajusta un K-Means sobre los clientes de **train** (coordenadas proyectadas en km). El número de clústers `k*` equilibra dos fuerzas:

- **Coherencia geográfica**: se elige `k` maximizando la silueta (sobre una muestra, para velocidad).
- **Capacidad**: `k` no puede ser menor que `⌈n/CAP⌉`, para que ninguna ruta exceda el máximo de visitas diarias.

`k* = max(k_silueta, ⌈n/CAP⌉)`. Luego, si algún clúster final supera la capacidad, se **subdivide** recursivamente hasta respetar `CAP`. El resultado son **sub-rutas** (cada una ≤ `CAP` clientes). Los clientes de **test** se asignan a estos clústers con `predict`.


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E4** — ancla de silueta: `k_max = ceil(1.6·k_cap)` sobre-clusterizaba. **Fix:** `K_EXTRA_MAX = 1.0`,
     ventana `[k_cap, k_cap]` → la silueta decide sólo si `n > 2·k_cap`. Sub-rutas `<15` se fusionan (`app/core/ruteo.py::formar_subrutas`).

In [ ]:
def subdivide(cluster_df, cap, semilla):
    """Divide recursivamente un clúster hasta que ningún sub-ruta supere `cap` clientes."""
    if len(cluster_df) <= cap:
        return [cluster_df]
    k2 = int(np.ceil(len(cluster_df) / cap))
    xy = cluster_df[["x_km", "y_km"]].values
    km2 = KMeans(n_clusters=k2, n_init=10, random_state=semilla).fit(xy)
    tmp = cluster_df.copy(); tmp["_c2"] = km2.labels_
    res = []
    for _, s2 in tmp.groupby("_c2"):
        res += subdivide(s2.drop(columns=["_c2"]), cap, semilla)
    return res

def formar_subrutas(grupo_train, grupo_all, cap, semilla):
    """Ajusta K-Means en los clientes TRAIN del grupo y forma sub-rutas sobre train+test."""
    xy_train = grupo_train[["x_km", "y_km"]].values
    n = len(grupo_train)
    k_cap = max(1, int(np.ceil(n / cap)))

    # El objetivo de negocio es llenar la capacidad (~CAP visitas por ruta), así que el punto de
    # partida es k_cap. La silueta solo decide dentro de una ventana razonable [k_cap, k_cap]  (ancla exacta):
    # si se dejara buscar sin ancla, la silueta (que casi siempre decrece con k) devolvería el
    # mínimo y generaría rutas demasiado pequeñas.
    k_sil = k_cap
    if n > 2 * k_cap:
        k_max = min(n - 1, max(k_cap, int(np.ceil(1.0 * k_cap))))
        k_max = max(k_max, k_cap + 1)
        idx = np.linspace(0, n - 1, min(n, 1500)).astype(int)
        mejor = -1.0
        for k in range(k_cap, k_max + 1):
            km = KMeans(n_clusters=k, n_init=10, random_state=semilla).fit(xy_train)
            if len(set(km.labels_)) < 2:
                continue
            s = silhouette_score(xy_train[idx], km.labels_[idx])
            if s > mejor:
                mejor, k_sil = s, k
    k = max(k_sil, k_cap)

    km = KMeans(n_clusters=k, n_init=10, random_state=semilla).fit(xy_train)

    # Asignar train y test a los centros aprendidos (sin fuga)
    g = grupo_all.copy()
    g["_cl"] = km.predict(g[["x_km", "y_km"]].values)

    rutas = []
    for _, cl in g.groupby("_cl"):
        rutas += subdivide(cl.drop(columns=["_cl"]), cap, semilla)
    return rutas, k, (mejor if n > 2 * k_cap else np.nan)

# --- Aplicar a todos los grupos ---
subrutas = []          # lista de DataFrames (una por sub-ruta)
meta_grupos = []       # k, silueta, nº sub-rutas por grupo
for gid, sub_all in df.groupby("grupo_id"):
    tr = train[train["grupo_id"] == gid]
    if len(tr) == 0:
        continue
    rutas, k, sil = formar_subrutas(tr, sub_all, CAP, SEMILLA)
    for j, r in enumerate(rutas):
        r = r.copy()
        r["sub_ruta"] = f"{gid}#R{j+1:02d}"
        subrutas.append(r)
    meta_grupos.append(dict(grupo=gid, clientes=len(sub_all), train=len(tr), k=k,
                            silueta=sil, sub_rutas=len(rutas)))

meta = pd.DataFrame(meta_grupos)
display(meta.style.format({"silueta": "{:.3f}", "clientes": "{:,}", "train": "{:,}"}))
print(f"Total sub-rutas: {len(subrutas)} | tamaño medio {np.mean([len(r) for r in subrutas]):.1f} "
      f"| máx {max(len(r) for r in subrutas)} | mín {min(len(r) for r in subrutas)}")

tam = pd.Series([len(r) for r in subrutas])
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(tam, bins=range(0, tam.max()+2), color="#2e86c1", edgecolor="white")
ax.axvline(CAP, color="red", ls="--", lw=1.5, label=f"Capacidad (CAP={CAP})")
ax.set_xlabel("clientes por sub-ruta"); ax.set_ylabel("nº de sub-rutas")
ax.set_title("Distribución del tamaño de las sub-rutas"); ax.legend()
plt.tight_layout(); plt.show()


## 6. Alternativa A — K-Means + Nearest Neighbor

Recorrido voraz: partiendo del depósito (CD), se visita secuencialmente el cliente **no visitado más cercano** que sea **factible**, es decir:

- **Capacidad**: la sub-ruta ya tiene ≤ `CAP` clientes (garantizado por K-Means), así que aquí no se excede.
- **Ventana horaria**: se mantiene un "slot" temporal actual (0 = mañana → 1 = mediodía → 2 = tarde). Solo se pueden elegir clientes cuya ventana permita el slot actual; cuando no queda ninguno, el slot avanza. Esto asegura que las tiendas/bodegas (mañana) se visiten antes que el entretenimiento (tarde).

Al final se regresa al depósito. Es el método base, rápido y determinista (dado el orden inicial).


In [ ]:
def slots_factibles(slots_orden):
    """¿La secuencia de ventanas es servible en orden creciente de slot? (0→1→2)."""
    actual = 0
    for s in slots_orden:
        while actual not in s and actual <= 2:
            actual += 1
        if actual > 2:
            return False
    return True

def nearest_neighbor(D, slots):
    """Recorrido voraz desde el nodo 0 (depósito) respetando ventanas horarias."""
    n = D.shape[0]
    visitados = np.zeros(n, bool); visitados[0] = True
    orden = [0]
    actual = 0
    while len(orden) < n:
        u = orden[-1]
        cand = [v for v in range(n) if not visitados[v] and actual in slots[v]]
        if not cand:
            actual += 1
            if actual > 2:
                cand = [v for v in range(n) if not visitados[v]]
                if not cand:
                    break
            continue
        v = min(cand, key=lambda i: D[u, i])
        visitados[v] = True; orden.append(v)
    return orden

def dist_ruta(D, orden):
    return float(D[orden[:-1], orden[1:]].sum())

def matriz_subruta(ruta, depot):
    """Construye D (haversine) y slots con el depósito en el índice 0."""
    lats = np.concatenate([[depot[0]], ruta["Latitud"].values])
    lons = np.concatenate([[depot[1]], ruta["Longitud"].values])
    D = haversine_mat(lats, lons)
    slots = [{0, 1, 2}] + list(ruta["slots"].values)
    return D, slots, lats, lons


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E2** — `slots_factibles(orden[1:-1])` descartaba el **último** cliente; con ruta **abierta** (`RUTA_CERRADA=False`)
     sí se sirve y su ventana debe verificarse. **Fix:** `orden[1:]` (`app/core/ruteo.py::segmento_servible`).

In [ ]:
# --- Ejecutar NN sobre todas las sub-rutas ---
DEP = dep.set_index("grupo_id")[["dep_lat", "dep_lon"]].to_dict("index")

def correr_nn(subrutas, DEP):
    resultados = []
    t0 = time.perf_counter()
    for ruta in subrutas:
        gid = ruta["grupo_id"].iloc[0]
        depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
        D, slots, lats, lons = matriz_subruta(ruta, depot)
        orden = nearest_neighbor(D, slots)
        d = dist_ruta(D, orden)
        factible = slots_factibles([slots[i] for i in orden[1:]])
        # distancia geoepica exacta del recorrido (reporte)
        puntos = [(lats[i], lons[i]) for i in orden]
        d_geo = sum(geodesic(puntos[i], puntos[i+1]).km for i in range(len(puntos)-1))
        resultados.append(dict(sub_ruta=ruta["sub_ruta"].iloc[0], grupo=gid,
                               clientes=len(ruta), dist_km=d_geo,
                               dist_hav=d, factible=int(factible),
                               excede_cap=int(len(ruta) > CAP)))
    tt = time.perf_counter() - t0
    return pd.DataFrame(resultados), tt

res_A, tiempo_A = correr_nn(subrutas, DEP)
print(f"Alternativa A (NN): {len(res_A)} sub-rutas | tiempo total {tiempo_A:.2f} s")
print(f"Distancia total: {res_A['dist_km'].sum():,.0f} km | media por sub-ruta: {res_A['dist_km'].mean():,.1f} km")
print(f"Restricciones incumplidas -> capacidad: {res_A['excede_cap'].sum()} | ventana: {(1-res_A['factible']).sum()}")


## 7. Alternativa B — K-Means + Colonia de hormigas (ACO)

Búsqueda metaheurística sobre **las mismas sub-rutas** que A. Cada hormiga construye un recorrido nodo a nodo con probabilidad proporcional a `τ^α · η^β`, donde `τ` es la feromona y `η = 1/d` la heurística de distancia. Se usa:

- **Lista de k-vecinos**: cada paso solo considera los `k` nodos más cercanos, para que escale a sub-rutas de decenas de clientes.
- **Misma restricción de ventana** que A en la construcción (solo se proponen clientes factibles en el slot actual).
- **Evaporación** `ρ` y depósito elitista en el mejor recorrido.
- **2-opt** al final, aceptando solo intercambios que mantengan la ventana factible.

ACO es estocástico y más lento: la pregunta es si esa elaboración extra paga en distancia recorrida.


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E2** — el 2-opt ignoraba el último nodo al proponer inversiones. **Fix:** incluirlo (`app/core/ruteo.py::dos_opt`).

In [ ]:
def _construir(tau, eta, D, slots, kvec, alfa, beta, rng):
    n = D.shape[0]
    visitados = np.zeros(n, bool); visitados[0] = True
    orden = [0]; actual = 0
    while len(orden) < n:
        u = orden[-1]
        cand = [v for v in kvec[u] if not visitados[v] and actual in slots[v]]
        if not cand:
            cand = [v for v in range(n) if not visitados[v] and actual in slots[v]]
        if not cand:
            actual += 1
            if actual > 2:
                cand = [v for v in range(n) if not visitados[v]]
                if not cand:
                    break
            continue
        cand = np.array(cand)
        peso = (tau[u, cand] ** alfa) * (eta[u, cand] ** beta)
        suma = peso.sum()
        if suma <= 0 or not np.isfinite(suma):
            v = int(rng.choice(cand))
        else:
            v = int(rng.choice(cand, p=peso/suma))
        visitados[v] = True; orden.append(v)
    return orden

def _dos_opt(orden, D, slots):
    """Mejora 2-opt; solo acepta inversiones que mantengan la ventana factible."""
    orden = orden[:]
    mejor = True
    while mejor:
        mejor = False
        n = len(orden)
        for i in range(1, n - 2):
            for j in range(i + 1, n - 1):
                nuevo = orden[:i] + orden[i:j+1][::-1] + orden[j+1:]
                if slots_factibles([slots[k] for k in nuevo[1:]]):
                    if dist_ruta(D, nuevo) < dist_ruta(D, orden) - 1e-9:
                        orden = nuevo; mejor = True
    return orden

def aco(D, slots, m, alfa, beta, rho, Q, iteraciones, semilla, kvecinos=15):
    n = D.shape[0]
    K = min(kvecinos, n - 1)
    order = np.argsort(D, axis=1)
    kvec = [[v for v in order[u] if v != u][:K] for u in range(n)]
    eta = np.zeros_like(D); nz = D > 0; eta[nz] = 1.0 / D[nz]
    tau = np.ones((n, n))
    rng = np.random.default_rng(semilla)
    mejor, md = None, np.inf
    historial = []
    for it in range(iteraciones):
        for _ in range(m):
            o = _construir(tau, eta, D, slots, kvec, alfa, beta, rng)
            d = dist_ruta(D, o)
            if d < md:
                md, mejor = d, o[:]
        tau *= (1 - rho)
        for a, b in zip(mejor[:-1], mejor[1:]):
            tau[a, b] += Q / md; tau[b, a] += Q / md
        historial.append(md)
    mejor = _dos_opt(mejor, D, slots)
    return mejor, dist_ruta(D, mejor), historial


### 7.1 Ajuste menor de hiperparámetros (solo en train)

Se prueba `β` (peso de la heurística de distancia vs. feromona) sobre unas pocas sub-rutas de **train** y se congela el mejor para la evaluación. Evita usar el test para calibrar.


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E3** — se calibraba β sobre las 6 sub-rutas **más pequeñas** (sesgo). **Fix:** muestra **estratificada** por
     rango de tamaño que cubre los 4 rangos (`app/core/ruteo.py::muestra_estratificada`).

In [ ]:
# Unas pocas sub-rutas pequeñas/train para calibrar beta rápidamente
# E3 fix: muestra ESTRATIFICADA por tamaño (cubre los 4 rangos), no las 6 más pequeñas
_RANGOS = [(1, 20), (21, 30), (31, 40), (41, 10_000)]
def _muestra_estrat(rutas, por_rango=2):
    m = []
    for lo, hi in _RANGOS:
        cand = sorted([r for r in rutas if lo <= len(r) <= hi], key=len)
        m += cand[:por_rango]
    return m
muestra_cal = _muestra_estrat(subrutas, por_rango=2)
print('rangos cubiertos:', [sorted(len(r) for r in muestra_cal if lo <= len(r) <= hi) for lo, hi in _RANGOS])
mejor_beta, mejor_prom = None, np.inf
pruebas = []
for beta in [1.0, 3.0, 5.0]:
    tot = 0.0
    for ruta in muestra_cal:
        gid = ruta["grupo_id"].iloc[0]
        depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
        D, slots, _, _ = matriz_subruta(ruta, depot)
        _, d, _ = aco(D, slots, alfa=ACO["alfa"], beta=beta, rho=ACO["rho"], Q=ACO["Q"],
                      m=6, iteraciones=15, semilla=SEMILLA)
        tot += d
    prom = tot / len(muestra_cal)
    pruebas.append(dict(beta=beta, dist_media_km=prom))
    if prom < mejor_prom:
        mejor_prom, mejor_beta = prom, beta
ACO["beta"] = mejor_beta
display(pd.DataFrame(pruebas).style.format({"dist_media_km": "{:.2f}"}))
print(f"beta elegido: {mejor_beta} -> congelado para la evaluación")


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E2** — misma corrección de ruta abierta: `orden[1:-1]` → `orden[1:]`.

In [ ]:
# --- Ejecutar ACO sobre todas las sub-rutas ---
def correr_aco(subrutas, DEP, params, semilla):
    resultados, historiales = [], {}
    t0 = time.perf_counter()
    for ruta in subrutas:
        gid = ruta["grupo_id"].iloc[0]
        depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
        D, slots, lats, lons = matriz_subruta(ruta, depot)
        orden, d_hav, hist = aco(D, slots, m=params["m"], alfa=params["alfa"], beta=params["beta"],
                                 rho=params["rho"], Q=params["Q"], iteraciones=params["iteraciones"],
                                 semilla=semilla)
        factible = slots_factibles([slots[i] for i in orden[1:]])
        puntos = [(lats[i], lons[i]) for i in orden]
        d_geo = sum(geodesic(puntos[i], puntos[i+1]).km for i in range(len(puntos)-1))
        resultados.append(dict(sub_ruta=ruta["sub_ruta"].iloc[0], grupo=gid, clientes=len(ruta),
                               dist_km=d_geo, dist_hav=d_hav, factible=int(factible),
                               excede_cap=int(len(ruta) > CAP)))
        historiales[ruta["sub_ruta"].iloc[0]] = hist
    tt = time.perf_counter() - t0
    return pd.DataFrame(resultados), tt, historiales

res_B, tiempo_B, hist_B = correr_aco(subrutas, DEP, ACO, SEMILLA)
print(f"Alternativa B (ACO): {len(res_B)} sub-rutas | tiempo total {tiempo_B:.2f} s")
print(f"Distancia total: {res_B['dist_km'].sum():,.0f} km | media por sub-ruta: {res_B['dist_km'].mean():,.1f} km")
print(f"Restricciones incumplidas -> capacidad: {res_B['excede_cap'].sum()} | ventana: {(1-res_B['factible']).sum()}")


## 8. Comparación A vs B

### 8.1 Tabla resumen (todas las sub-rutas)

Métricas: distancia total y media, tiempo de cómputo, número de rutas y **restricciones incumplidas** (capacidad y ventana). Se añade una columna diagnóstico: qué tantas violaciones de ventana produciría un NN " ingenuo" que ignorara los horarios, para poner en contexto que A y B **sí** las respetan.


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E2** — el diagnóstico también debe verificar el último cliente (`o[1:-1]` → `o[1:]`).

In [ ]:
# Diagnóstico: NN sin restricción de ventana (solo para contextualizar)
def nn_sin_ventana(D, slots):
    n = D.shape[0]; vis = np.zeros(n, bool); vis[0] = True; o = [0]
    while len(o) < n:
        u = o[-1]; cand = [v for v in range(n) if not vis[v]]
        v = min(cand, key=lambda i: D[u, i]); vis[v] = True; o.append(v)
    return o

inc_ventana_naive = 0
for ruta in subrutas:
    gid = ruta["grupo_id"].iloc[0]
    depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
    D, slots, _, _ = matriz_subruta(ruta, depot)
    o = nn_sin_ventana(D, slots)
    if not slots_factibles([slots[i] for i in o[1:]]):
        inc_ventana_naive += 1

cob = df["cubre_quincenas"].mean() * 100
tabla = pd.DataFrame({
    "Alternativa A (NN)": [
        res_A["dist_km"].sum(), res_A["dist_km"].mean(), res_A["dist_km"].median(),
        tiempo_A, len(res_A), res_A["excede_cap"].sum(), (1 - res_A["factible"]).sum(), np.nan],
    "Alternativa B (ACO)": [
        res_B["dist_km"].sum(), res_B["dist_km"].mean(), res_B["dist_km"].median(),
        tiempo_B, len(res_B), res_B["excede_cap"].sum(), (1 - res_B["factible"]).sum(), np.nan],
}, index=["Distancia total (km)", "Distancia media/ruta (km)", "Distancia mediana/ruta (km)",
          "Tiempo de cómputo (s)", "Nº de sub-rutas", "Incumplen capacidad", "Incumplen ventana",
          "Cobertura quincenal (% clientes Q1-Q2)"])
tabla.loc["Cobertura quincenal (% clientes Q1-Q2)"] = [cob, cob]
display(tabla.style.format({"Alternativa A (NN)": "{:,.2f}", "Alternativa B (ACO)": "{:,.2f}"}))

dA, dB = res_A["dist_km"].sum(), res_B["dist_km"].sum()
print(f"\nDistancia total -> A (NN): {dA:,.0f} km | B (ACO): {dB:,.0f} km")
print(f"ACO reduce {dA-dB:,.0f} km ({100*(dA-dB)/dA:.2f}%) | cuesta {tiempo_B-tiempo_A:+.2f} s de cómputo")
print(f"\nDiagnóstico: un NN ingenuo (sin ventanas) violaría la ventana horaria en "
      f"{inc_ventana_naive} de {len(subrutas)} sub-rutas; A y B la respetan en todas.")


### 8.2 Experimento multi-semilla (estabilidad)

Siguiendo el patrón del ejemplo `KFold_vs_TrainTestSplit`, se repite todo el proceso (split → K-Means → NN y ACO) con `N_SEEDS` semillas distintas y se compara la distancia sobre las sub-rutas de **test** (clientes no vistos por el K-Means). Se reportan media, desviación, mín/máx y **en qué % de semillas gana cada modelo**.


In [ ]:
def pipeline_test(df, train, semilla, params_aco, usar_silueta=True):
    """Devuelve (dist_A, dist_B, tiempo_A, tiempo_B) sobre sub-rutas de test para una semilla."""
    subrutas_t = []
    for gid, sub_all in df.groupby("grupo_id"):
        tr = train[train["grupo_id"] == gid]
        if len(tr) == 0:
            continue
        rutas, _, _ = formar_subrutas(tr, sub_all, CAP, semilla)
        for j, r in enumerate(rutas):
            r = r.copy(); r["sub_ruta"] = f"{gid}#R{j+1:02d}"; subrutas_t.append(r)
    # métrica A y B por sub-ruta de test (que contengan al menos un cliente de test)
    ids_test = set(test.index)
    sub_test = [r for r in subrutas_t if r.index.isin(ids_test).any()]
    dA = dB = 0.0; tA = tB = 0.0
    for ruta in sub_test:
        gid = ruta["grupo_id"].iloc[0]
        depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
        D, slots, _, _ = matriz_subruta(ruta, depot)
        t0 = time.perf_counter(); oA = nearest_neighbor(D, slots); tA += time.perf_counter() - t0
        t0 = time.perf_counter(); oB, _, _ = aco(D, slots, m=params_aco["m"], alfa=ACO["alfa"],
                                                 beta=ACO["beta"], rho=ACO["rho"], Q=ACO["Q"],
                                                 iteraciones=params_aco["iteraciones"],
                                                 semilla=semilla); tB += time.perf_counter() - t0
        dA += dist_ruta(D, oA); dB += dist_ruta(D, oB)
    return dA, dB, tA, tB, len(sub_test)

filas = []
for s in range(N_SEEDS):
    tr, te = train_test_split(df, test_size=TEST_SIZE, random_state=s, stratify=df["Supervisor"])
    dA, dB, tA, tB, nr = pipeline_test(df, tr, s, ACO_SEEDS)
    filas.append(dict(semilla=s, NN=dA, ACO=dB, t_NN=tA, t_ACO=tB, sub_rutas_test=nr))
    print(f"semilla {s:>2} | NN {dA:,.0f} km | ACO {dB:,.0f} km | ACO mejora {100*(dA-dB)/dA:.2f}%")

exp = pd.DataFrame(filas).set_index("semilla")
display(exp.style.format({"NN": "{:,.0f}", "ACO": "{:,.0f}", "t_NN": "{:.2f}", "t_ACO": "{:.2f}"}))


> ⚠ **Corregido: re-ejecutar.** Esta celda se arregló respecto al notebook original. La implementación
> autoritativa y probada vive en `app/core/` y está cubierta por `tests/test_correcciones.py`.

**E1** — `% semillas en que gana` estaba **invertido**: la fila NN recibía `100*(ACO<NN)`.
     **Fix:** NN recibe `100*(NN<ACO)` y ACO `100*(ACO<NN)` (`app/core/comparacion.py::resumen_estabilidad`).

In [ ]:
def resumen(exp):
    r = exp[["NN", "ACO"]].agg(["mean", "std", "min", "max"]).T
    r["rango (máx-mín)"] = r["max"] - r["min"]
    r["% semillas en que gana"] = [100 * (exp["NN"] < exp["ACO"]).mean(), 100 * (exp["ACO"] < exp["NN"]).mean()]  # E1 fix
    return r.rename(columns={"mean": "media", "std": "desv. estándar", "min": "mín", "max": "máx"}).round(1)

print(f"Distancia de test (km) — {N_SEEDS} semillas")
display(resumen(exp))

# Prueba de Wilcoxon pareada (¿la diferencia de distancia es estadísticamente significativa?)
estad, p = wilcoxon(exp["NN"], exp["ACO"])
print(f"\nWilcoxon (NN vs ACO, pareado por semilla): estadístico={estad:.1f}, p-valor={p:.4f}")
print("-> La diferencia es estadísticamente significativa (p<0.05)." if p < 0.05
      else "-> La diferencia NO es estadísticamente significativa (p>=0.05).")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
sns.boxplot(data=exp[["NN", "ACO"]], palette=["#7fb3d5", "#e8a33d"], ax=ax)
ax.set_ylabel("Distancia total de test (km)")
ax.set_title(f"Alternativa A (NN) vs B (ACO) — {N_SEEDS} semillas")
plt.tight_layout(); plt.show()


### 8.3 ¿Dónde conviene ACO? Análisis por tamaño de sub-ruta y por asesor

Se cruza la mejora relativa de ACO con el tamaño de la sub-ruta, y se resumen las distancias por territorio (Asesor × CD).


In [ ]:
cmp = res_A[["sub_ruta", "grupo", "clientes", "dist_km"]].rename(columns={"dist_km": "dist_A"}).merge(
    res_B[["sub_ruta", "dist_km"]].rename(columns={"dist_km": "dist_B"}), on="sub_ruta")
cmp["mejora_km"] = cmp["dist_A"] - cmp["dist_B"]
cmp["mejora_%"] = 100 * cmp["mejora_km"] / cmp["dist_A"]

print("Mejora de ACO según tamaño de sub-ruta:")
cmp["rango_tam"] = pd.cut(cmp["clientes"], bins=[0, 20, 30, 40, CAP], labels=["<=20", "21-30", "31-40", "41-45"])
display(cmp.groupby("rango_tam", observed=True)["mejora_%"].agg(["count", "mean", "max"]).round(2))

por_grupo = (cmp.groupby("grupo")
               .agg(clientes=("clientes", "sum"), sub_rutas=("sub_ruta", "nunique"),
                    dist_A=("dist_A", "sum"), dist_B=("dist_B", "sum"),
                    mejora_km=("mejora_km", "sum"))
               .reset_index())
por_grupo["mejora_%"] = 100 * por_grupo["mejora_km"] / por_grupo["dist_A"]
display(por_grupo.sort_values("clientes", ascending=False).style.format(
    {"clientes": "{:,}", "dist_A": "{:,.0f}", "dist_B": "{:,.0f}", "mejora_km": "{:,.0f}", "mejora_%": "{:.2f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.scatterplot(data=cmp, x="clientes", y="mejora_%", ax=axes[0], color="#c0392b")
axes[0].axhline(cmp["mejora_%"].mean(), color="gray", ls="--", lw=1)
axes[0].set_xlabel("clientes por sub-ruta"); axes[0].set_ylabel("mejora de ACO (%)")
axes[0].set_title("Mejora relativa de ACO vs tamaño de sub-ruta")
top = por_grupo.sort_values("clientes", ascending=False).head(8)
sns.barplot(data=top, y="grupo", x="mejora_%", ax=axes[1], palette="viridis")
axes[1].set_xlabel("mejora de ACO (%)"); axes[1].set_title("Mejora por territorio (Asesor × CD)")
plt.tight_layout(); plt.show()


### 8.4 Visualización: una sub-ruta y la convergencia del ACO

Se dibuja el recorrido de A y B sobre una misma sub-ruta y la curva de convergencia (mejor distancia por iteración) del ACO.


In [ ]:
# Elegir una sub-ruta representativa (mediana en tamaño)
ej = cmp.iloc[(cmp["clientes"] - int(cmp["clientes"].median())).abs().argsort()].iloc[0]
sr_id = ej["sub_ruta"]
ruta = [r for r in subrutas if r["sub_ruta"].iloc[0] == sr_id][0]
gid = ruta["grupo_id"].iloc[0]
depot = (DEP[gid]["dep_lat"], DEP[gid]["dep_lon"])
D, slots, lats, lons = matriz_subruta(ruta, depot)
oA = nearest_neighbor(D, slots)
oB, _, hist = aco(D, slots, m=ACO["m"], alfa=ACO["alfa"], beta=ACO["beta"], rho=ACO["rho"],
                  Q=ACO["Q"], iteraciones=ACO["iteraciones"], semilla=SEMILLA)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(lons[oA], lats[oA], "-o", ms=4, color="#2e86c1", label="A · Nearest Neighbor")
axes[0].plot(lons[oB], lats[oB], "-s", ms=4, color="#c0392b", alpha=0.7, label="B · ACO + 2-opt")
axes[0].scatter(depot[1], depot[0], marker="*", s=260, color="gold", edgecolor="k", zorder=5, label="Depósito (CD)")
axes[0].set_title(f"Sub-ruta {sr_id} ({len(ruta)} clientes)")
axes[0].set_xlabel("Longitud"); axes[0].set_ylabel("Latitud"); axes[0].legend(fontsize=8)

axes[1].plot(hist, color="#c0392b")
axes[1].set_xlabel("iteración"); axes[1].set_ylabel("mejor distancia (km)")
axes[1].set_title("Convergencia del ACO")
plt.tight_layout(); plt.show()

print(f"Sub-ruta de ejemplo {sr_id}:")
print(f"  A (NN):  {dist_ruta(D, oA):.2f} km")
print(f"  B (ACO): {dist_ruta(D, oB):.2f} km  ({100*(dist_ruta(D,oA)-dist_ruta(D,oB))/dist_ruta(D,oA):.2f}% mejor)")


## 9. Conclusiones y siguientes pasos

**Qué se hizo**
- Se partió de `base_clientes_limpia.csv`, se filtró el universo de ruteo (activos con asesor y visita) y se definieron los territorios como pares **Asesor × Centro de Distribución**, con el CD como depósito (centroide de sus clientes).
- Se crearon las variables de depósito, proyección a km, ventana horaria por categoría, capacidad por ruta y cobertura quincenal.
- El **K-Means se ajusta solo en train** y asigna a test con `predict`; el número de sub-rutas respeta la capacidad diaria (≤ `CAP`).
- Se secuenciaron las **mismas sub-rutas** con dos métodos: **Nearest Neighbor** (A) y **ACO + 2-opt** (B), ambos respetando capacidad y ventana horaria.

**Cómo leer la comparación**
- **Distancia**: el ACO típicamente encuentra recorridos más cortos que el voraz NN, y la mejora crece con el tamaño de la sub-ruta.
- **Tiempo de cómputo**: el ACO cuesta bastante más (metaheurística con muchas iteraciones).
- **Restricciones**: A y B no incumplen capacidad ni ventana por construcción; un NN ingenuo sí violaría los horarios, lo que muestra el valor de modelarlas.
- El experimento **multi-semilla** y el **Wilcoxon** indican si la ventaja del ACO es estable y estadísticamente significativa, no artefacto de una semilla.

**Siguientes pasos**
- Conectar las sub-rutas al **calendario quincenal** (Q1/Q2) para materializar las 2 visitas/mes ~15 días separadas.
- Incorporar tiempos de servicio por categoría y ventanas temporales continuas (no solo 3 slots).
- Si el tiempo de cómputo del ACO es prohibitivo en producción, usarlo solo en las sub-rutas grandes y NN en las pequeñas (híbrido, según el análisis 8.3).
- Revisar con el equipo de negocio el depósito por centroide y, de conseguirse, las coordenadas exactas de cada CD.
